# Exercício Complementar — Classificação com LSTM (SLR)

Notebook pronto para Google Colab.

Arquitetura solicitada:
- Embedding: 300 dimensões
- 2 camadas LSTM com 300 neurônios
- 1 camada Dense com 300 neurônios
- Saída com 1 neurônio (sigmoid)
- Sequências com tamanho máximo de 200 tokens
- Treino por 10 épocas


In [ ]:
# Dependências (opcional no Colab; geralmente já vêm instaladas)
!pip -q install pandas scikit-learn tensorflow

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

In [ ]:
# Download/carregamento do dataset
DATA_URL = 'https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/slr.csv'
df = pd.read_csv(DATA_URL)
df.head()

In [ ]:
# Validação das colunas esperadas
required = {'title-abstract', 'label'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Colunas ausentes: {missing}')

X = df['title-abstract'].fillna('').astype(str)
y_raw = df['label'].astype(str)

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

if len(encoder.classes_) != 2:
    raise ValueError(f'Este notebook foi configurado para classificação binária. Classes: {list(encoder.classes_)}')

print('Classes:', list(encoder.classes_))
print('Total de amostras:', len(X))

In [ ]:
# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Tokenização e padding
MAX_WORDS = 50000
MAX_TOKENS = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_TOKENS, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_TOKENS, padding='post', truncating='post')

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
print('Vocabulário usado:', vocab_size)
print('Shape treino:', X_train_pad.shape)
print('Shape teste:', X_test_pad.shape)

In [ ]:
# Modelo LSTM conforme enunciado
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=300, input_length=MAX_TOKENS),
    LSTM(300, return_sequences=True),
    LSTM(300),
    Dense(300, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Treino (10 épocas)
history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Avaliação no conjunto de teste
y_proba = model.predict(X_test_pad).ravel()
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
precision, recall, fscore, support = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1], zero_division=0
)

print(f'Acurácia geral: {acc:.4f}')
print('\nMétricas por classe:')
for i, class_name in enumerate(encoder.classes_):
    print(
        f"- Classe '{class_name}': Precision={precision[i]:.4f}, Recall={recall[i]:.4f}, F-Score={fscore[i]:.4f}, Support={support[i]}"
    )